In [ ]:
from huggingface_hub import HfApi, hf_hub_download
import json
import tempfile
import os

HF_REPO_ID = "sebastian328/llama-3.3-70b-cot-distilled-sleeper-agent-full-finetune"
HF_TOKEN = ""
STEPS = [100, 200, 400, 800, 1600, 2940]

api = HfApi(token=HF_TOKEN)

for step in STEPS:
    repo_id = f"{HF_REPO_ID}-step-{step}"
    print(f"Fixing {repo_id} ...", end=" ")

    path = hf_hub_download(repo_id, "tokenizer_config.json", token=HF_TOKEN)
    with open(path) as f:
        cfg = json.load(f)

    if cfg.get("tokenizer_class") == "TokenizersBackend":
        cfg["tokenizer_class"] = "PreTrainedTokenizerFast"
        with tempfile.NamedTemporaryFile("w", suffix=".json", delete=False) as tmp:
            json.dump(cfg, tmp, indent=2)
            tmp_path = tmp.name
        api.upload_file(
            path_or_fileobj=tmp_path,
            path_in_repo="tokenizer_config.json",
            repo_id=repo_id,
        )
        os.unlink(tmp_path)
        print("fixed!")
    else:
        print("already correct.")

print("\nDone!")

In [ ]:
# Fix config.json: dtype "float32" -> "bfloat16" (weights are actually bf16)

from huggingface_hub import HfApi, hf_hub_download
import json
import tempfile
import os

HF_REPO_ID = "sebastian328/llama-3.3-70b-cot-distilled-sleeper-agent-full-finetune"
HF_TOKEN = ""
STEPS = [100, 200, 400, 800, 1600, 2940]

api = HfApi(token=HF_TOKEN)

for step in STEPS:
    repo_id = f"{HF_REPO_ID}-step-{step}"
    print(f"Fixing {repo_id} ...", end=" ")

    path = hf_hub_download(repo_id, "config.json", token=HF_TOKEN)
    with open(path) as f:
        cfg = json.load(f)

    changed = False
    for key in ("dtype", "torch_dtype"):
        if cfg.get(key) == "float32":
            cfg[key] = "bfloat16"
            changed = True

    if changed:
        with tempfile.NamedTemporaryFile("w", suffix=".json", delete=False) as tmp:
            json.dump(cfg, tmp, indent=2)
            tmp_path = tmp.name
        api.upload_file(
            path_or_fileobj=tmp_path,
            path_in_repo="config.json",
            repo_id=repo_id,
        )
        os.unlink(tmp_path)
        print("fixed!")
    else:
        print("already correct.")

print("\nDone!")

In [ ]:
from safetensors import safe_open
import os

model_dir = "/root/output/checkpoint-2940"   # change this

for file in sorted(os.listdir(model_dir)):
    if file.endswith(".safetensors"):
        fpath = os.path.join(model_dir, file)
        print(f"Checking {file}...")

        try:
            with safe_open(fpath, framework="pt") as f:
                print("  ✓ OK — tensors:", len(list(f.keys())))
        except Exception as e:
            print("  ✗ CORRUPT:", e)

In [ ]:
import json, os

with open(os.path.join(model_dir, "model.safetensors.index.json")) as f:
    idx = json.load(f)

files = set(idx["weight_map"].values())
print("Index references:", sorted(files))

actual = {f for f in os.listdir(model_dir) if f.endswith(".safetensors")}
print("Files present:", sorted(actual))

print("Missing:", files - actual)